## 1. Imports & Load Raw Data

In [187]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os.path as path

In [188]:
df_wineRead = pd.read_csv('dataset/winequality-red.csv',sep=';')
df_wineWhite  = pd.read_csv('dataset/winequality-white.csv',sep=';')
df = pd.concat([df_wineRead,df_wineWhite],ignore_index=True)

y_target = 'quality'
df.head(5)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


## 2. Handle Missing / Invalid Values

In [189]:
missing_feature = df.columns[df.isnull().any()].to_list()
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Missing Feature:\n{missing_feature}')

Missing values: 0
Duplicates: 1179
Missing Feature:
[]


In [190]:
for col in missing_feature:
    missing_percentage = (df[col].isnull().sum() / len(df)) * 100
    if missing_percentage > 5.0 :
        df = df.drop(columns=col)
    else:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            skewness = df[col].skew()
            if abs(skewness) < 0.5:
                df[col] = df[col].fillna(df[col].mean()).round(2)
            else:
                df[col] = df[col].fillna(df[col].median()).round(2)

print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Missing Feature:\n{missing_feature}')

Missing values: 0
Missing Feature:
[]


## 3. Handling Duplicated

In [191]:
df = df.drop_duplicates(keep='last')

jumlah_duplikat = df.duplicated().sum()
print(f"Jumlah data duplikat: {jumlah_duplikat}")

Jumlah data duplikat: 0


## 4. Analisis & Handling Outliers

In [192]:
feature_numerik = df.select_dtypes(include=np.number).columns.drop(y_target)
Q1 = df[feature_numerik].quantile(0.25)
Q3 = df[feature_numerik].quantile(0.75)
IQR = Q3-Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sebelum Dihapus: {outliers.shape[0]}")

#delete outliers
df = df.loc[((df[feature_numerik] >= lower_bound) & (df[feature_numerik] <= upper_bound)).all(axis=1)]
outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sesudah Dihapus: {outliers.shape[0]}")

Jumlah Outlier Sebelum Dihapus: 1094
Jumlah Outlier Sesudah Dihapus: 0


## 5. Feature Engineering

In [193]:
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
5,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5
6,7.9,0.60,0.06,1.6,0.069,15.0,59.0,0.9964,3.30,0.46,9.4,5
7,7.3,0.65,0.00,1.2,0.065,15.0,21.0,0.9946,3.39,0.47,10.0,7
8,7.8,0.58,0.02,2.0,0.073,9.0,18.0,0.9968,3.36,0.57,9.5,7
10,6.7,0.58,0.08,1.8,0.097,15.0,65.0,0.9959,3.28,0.54,9.2,5


### 5.1 Encoding to Text Feautures Target

In [194]:
df[y_target] = df[y_target].apply(lambda x: 'GoodQuality' if x > 5.1 else 'BadQuality')

### 5.2 Encoding Bins in Features Numeric

In [195]:
for col in feature_numerik:
    df[f'{col}_category'] = pd.qcut(df[col],q=3,labels=['Low','Medum','High'],duplicates='drop')

## 6.Save Cleaned Dataset

In [198]:
file_path = 'dataset/winequality_concat_CLEANING.csv'

if not path.exists(file_path):
    df.to_csv(file_path, index=False)
    print('File belum ada. Berhasil menyimpan dataset baru!')
else:
    print('File sudah ada. Proses penyimpanan CSV dilewati (skip)')

df.head()

File belum ada. Berhasil menyimpan dataset baru!


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,...,volatile acidity_category,citric acid_category,residual sugar_category,chlorides_category,free sulfur dioxide_category,total sulfur dioxide_category,density_category,pH_category,sulphates_category,alcohol_category
5,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,...,High,Low,Low,High,Low,Low,High,High,High,Low
6,7.9,0.60,0.06,1.6,0.069,15.0,59.0,0.9964,3.30,0.46,...,High,Low,Low,High,Low,Low,High,High,Medum,Low
7,7.3,0.65,0.00,1.2,0.065,15.0,21.0,0.9946,3.39,0.47,...,High,Low,Low,High,Low,Low,Medum,High,Medum,Medum
8,7.8,0.58,0.02,2.0,0.073,9.0,18.0,0.9968,3.36,0.57,...,High,Low,Low,High,Low,Low,High,High,High,Low
10,6.7,0.58,0.08,1.8,0.097,15.0,65.0,0.9959,3.28,0.54,...,High,Low,Low,High,Low,Low,High,Medum,Medum,Low
